# Are the teacher labels faithful? (Blockers 3 + 4)

Everything so far has been plumbing. This notebook asks the question that decides whether the
whole Stage-2b teacher branch is real:

> **Does the attention-derived importance map actually track what the model used to answer?**

## Ground truth

Mask image patch `i`, re-run the frozen VLM, and measure how much the log-probability of the
gold answer drops. That drop **is** the importance of patch `i` — no attention heuristics involved.
This is Method B (leave-one-out) from the FRM spec, and it is the only ground truth available.

## What gets scored against it

| label | rows summed | correction |
|---|---|---|
| `imp_q` raw | question tokens | none |
| `imp_q` subtract | question tokens | probability-space, clamped |
| `imp_q` pmi | question tokens | log-space |
| `imp_a` raw / subtract / pmi | **answer** tokens | — |
| `imp_context` | answer minus question | log-space |
| `center` | — | constant baseline |
| `gaze proximity` | — | the eccentricity rival |

That settles three open items at once:

* **Blocker 4** — is any of this a real attribution, or just a saliency map?
* **Blocker 3** — `imp_answer` vs `imp_question` (the spec's training label vs its control).
* **subtract vs pmi** — judged against real attribution instead of a synthetic I designed.

> **Runtime:** GPU. Needs `MyDrive/wearvqa_gaze_only`. First run downloads ~4.5 GB of weights.
> Budget roughly 10-20 minutes for 20 examples.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip -q install -U "transformers>=4.49" accelerate huggingface_hub safetensors pillow num2words matplotlib scipy
!rm -rf /content/text_vision_attention_map
!git clone -q https://github.com/shubhamOjha1000/text_vision_attention_map.git /content/text_vision_attention_map
%cd /content/text_vision_attention_map

In [ ]:
import importlib.util, os, sys, math, glob, json, time
import numpy as np
import torch
import matplotlib.pyplot as plt
from collections import defaultdict
from scipy.stats import spearmanr

sys.path.insert(0, os.getcwd())

def _load(mod, rel):
    spec = importlib.util.spec_from_file_location(mod, os.path.join(os.getcwd(), rel))
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m

S = _load("probe_smolvlm", "tests/probe_smolvlm.py")
import rater_selection as RS
import visual_selection as VS

assert hasattr(VS, "teacher_label"), "clone is stale - visual_selection has no teacher_label"

# ---------------- config ----------------
DATA_ROOT   = "/content/drive/MyDrive/wearvqa_gaze_only"
CACHE       = "/content/drive/MyDrive/wearvqa_faithfulness.pt"
MODEL_ID    = "HuggingFaceTB/SmolVLM2-2.2B-Instruct"
N_PER_TYPE  = 2      # 2 x 10 types = 20 examples
GROUP       = 1      # 1 = ablate each patch alone (81 passes); 2 = 2x2 blocks (25 passes, ~3x faster)
DROP_SINK_K = 3

assert os.path.isdir(DATA_ROOT), f"not found: {DATA_ROOT}"

model, processor, device = S._load_smolvlm(MODEL_ID)
tokenizer = processor.tokenizer
print("device:", device, "| image splitting:",
      getattr(processor.image_processor, "do_image_splitting", "?"))

## 2. Load the dataset

WEAR-VQA carries the gold answer in the `response` field, so no generation is needed and the
scoring target is deterministic.

In [ ]:
TYPES = sorted(d for d in os.listdir(DATA_ROOT) if os.path.isdir(os.path.join(DATA_ROOT, d)))
samples = []
for t in TYPES:
    for jp in sorted(glob.glob(os.path.join(DATA_ROOT, t, "*.json")))[:N_PER_TYPE]:
        meta = json.load(open(jp))
        img_path = jp[:-5] + ".jpg"
        if not os.path.exists(img_path) or "gaze" not in meta or not meta.get("response"):
            continue
        samples.append(dict(type=t, img_path=img_path,
                            question=meta["question"], answer=meta["response"],
                            gaze=meta["gaze"], rationale=meta["gaze"].get("rationale", "")))

print(f"types: {len(TYPES)} | usable samples: {len(samples)}")
print("\nexample:")
print("  Q:", samples[0]["question"])
print("  A:", samples[0]["answer"])

## 3. Machinery

`build_inputs` appends the gold answer after the generation prompt, so the answer tokens sit in
the sequence and can be both **scored** (section 4) and **read as attention rows** (section 5).

Patches are ablated by zeroing their entry in `attention_mask`: the token stays in place but no
other token can attend to it, so its information is removed from the forward pass.

In [ ]:
def build_inputs(image, question, answer):
    """-> (inputs, n_prompt). Tokens from n_prompt onward are the answer."""
    messages = [{"role": "user", "content": [{"type": "image"},
                                             {"type": "text", "text": question}]}]
    prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
    full = processor(text=prompt + answer, images=[image], return_tensors="pt").to(device)
    only = processor(text=prompt, images=[image], return_tensors="pt")
    n_prompt = int(only["input_ids"].shape[1])
    # the prompt must be a strict prefix of the scored sequence
    assert torch.equal(full["input_ids"][0, :n_prompt].cpu(), only["input_ids"][0]), \
        "prompt is not a token-prefix of prompt+answer"
    assert n_prompt < full["input_ids"].shape[1], "answer tokenised to nothing"
    return full, n_prompt


@torch.no_grad()
def answer_logprob(inp, n_prompt, attention_mask=None):
    """sum of log P(answer token | everything before it)."""
    kw = dict(inp)
    if attention_mask is not None:
        kw["attention_mask"] = attention_mask
    logits = model(**kw).logits[0].float()
    lp = torch.log_softmax(logits[:-1], dim=-1)
    tgt = inp["input_ids"][0, 1:]
    tok = lp.gather(-1, tgt[:, None]).squeeze(-1)
    return float(tok[n_prompt - 1:].sum())


def patch_groups(L_v, g):
    """[[i], [j], ...] single patches, or g x g blocks on the sqrt(L_v) grid."""
    G = int(round(math.sqrt(L_v)))
    if g <= 1:
        return [[i] for i in range(L_v)]
    out = []
    for r0 in range(0, G, g):
        for c0 in range(0, G, g):
            out.append([r * G + c for r in range(r0, min(r0 + g, G))
                                  for c in range(c0, min(c0 + g, G))])
    return out


def loo_drops(inp, n_prompt, img_pos, group=1):
    """GROUND TRUTH: drop in answer logprob when each patch is ablated. -> (base, [L_v])."""
    base = answer_logprob(inp, n_prompt)
    L_v = len(img_pos)
    drops = torch.zeros(L_v)
    for grp in patch_groups(L_v, group):
        am = inp["attention_mask"].clone()
        am[0, img_pos[grp]] = 0
        drops[grp] = (base - answer_logprob(inp, n_prompt, am)) / len(grp)
    return base, drops


def raw_scores_for(inp):
    """Capture pre-softmax decoder attention for the FULL (question+answer) sequence."""
    patched = S._patch_eager_globals(S._make_raw_capturing_eager(None))
    try:
        with torch.no_grad():
            model(**inp)
    finally:
        S._unpatch_eager_globals(patched)
    L = int(inp["input_ids"].shape[1])
    raw = {}
    for m in model.modules():
        r = getattr(m, "_raw_attn_scores", None)
        if r is not None and r.shape[-1] == L and r.shape[-2] == L:
            raw[int(getattr(m, "layer_idx", len(raw)))] = r[0].float()
        for attr in ("_raw_attn_scores", "_post_attn"):
            if hasattr(m, attr):
                delattr(m, attr)
    assert raw, "no language-model attention captured"
    return raw

print("ok")

## 4 + 5. The expensive pass

Per example: 1 baseline forward, then one forward per patch group (the ground truth), then one
attention-capturing forward that yields both `imp_question` and `imp_answer`.

Results are cached to Drive, so a disconnect does not cost you the whole run.

In [ ]:
def process(s):
    img = S.load_image(s["img_path"])
    inp, n_prompt = build_inputs(img, s["question"], s["answer"])

    ids = inp["input_ids"][0].cpu()
    img_id = S._find_image_token_id(model, processor)
    img_mask = ids == img_id
    pad = tokenizer.pad_token_id
    txt_mask = (ids != img_id) & (ids != (pad if pad is not None else -10**9))
    img_pos = torch.nonzero(img_mask).squeeze(-1)

    # ---- ground truth ----
    base_lp, drops = loo_drops(inp, n_prompt, img_pos, GROUP)

    # ---- attention labels, question rows AND answer rows ----
    raw = raw_scores_for(inp)
    maps, tpos, _ = RS.sliced_maps_from_full(raw, img_mask, txt_mask)
    tt = tokenizer.convert_ids_to_tokens(ids[tpos].tolist())

    rr_q = RS.select_important_text_tokens(maps, text_tokens=tt, tokenizer=tokenizer,
                                           question=s["question"], pct=0.5)
    content = RS.content_text_mask(tt, tokenizer)
    is_ans = torch.tensor([int(p) >= n_prompt for p in tpos.tolist()])
    ans_mask = content & is_ans
    if int(ans_mask.sum()) == 0:
        ans_mask = is_ans                       # fall back to every answer token

    imp_q, *_ = VS.image_importance(maps, rr_q.rater_mask)
    imp_a, *_ = VS.image_importance(maps, ans_mask)

    del raw, maps
    return dict(type=s["type"], question=s["question"], answer=s["answer"],
                gaze=s["gaze"], rationale=s["rationale"], img_path=s["img_path"],
                base_logp=base_lp, drops=drops, imp_q=imp_q, imp_a=imp_a,
                raters_q=rr_q.kept_tokens(tt),
                raters_a=[t for t, m in zip(tt, ans_mask.tolist()) if m][:12],
                n_answer_rows=int(ans_mask.sum()))


if os.path.exists(CACHE):
    data = torch.load(CACHE, weights_only=False)
    print(f"loaded {len(data)} cached results from {CACHE}")
else:
    data, t0 = [], time.time()
    for i, s in enumerate(samples):
        try:
            data.append(process(s))
        except Exception as e:
            print(f"  [skip {i} {s['type']}] {type(e).__name__}: {e}")
            continue
        if i == 0:
            per = time.time() - t0
            print(f"first example took {per:.1f}s -> ETA {per * len(samples) / 60:.1f} min")
        if (i + 1) % 5 == 0:
            print(f"  {i + 1}/{len(samples)}  ({(time.time() - t0) / 60:.1f} min)")
    torch.save(data, CACHE)
    print(f"\ndone in {(time.time() - t0) / 60:.1f} min | cached -> {CACHE}")

L_v = data[0]["drops"].numel()
G = int(round(math.sqrt(L_v)))
print(f"\n{len(data)} examples | L_v={L_v} ({G}x{G}) | "
      f"mean answer logprob {np.mean([d['base_logp'] for d in data]):.2f}")
print("answer rows per example:", [d["n_answer_rows"] for d in data][:10], "...")

### Sanity check the ground truth first

If ablating patches does not move the answer logprob at all, there is nothing for any label to
predict and every correlation below is meaningless. Check that the drops have real spread and are
not dominated by a single patch.

In [ ]:
allsd = np.array([float(d["drops"].std()) for d in data])
allmx = np.array([float(d["drops"].max()) for d in data])
frac  = np.array([float(d["drops"].max() / (d["drops"].abs().sum() + 1e-9)) for d in data])

print(f"per-example drop std   : mean {allsd.mean():.4f}  (0 -> ablation does nothing)")
print(f"per-example max drop   : mean {allmx.mean():.4f} nats")
print(f"top patch's share      : mean {frac.mean():.1%} of total |drop|")
print(f"examples with std < 1e-3: {(allsd < 1e-3).sum()} / {len(data)}")

plt.figure(figsize=(5, 3))
plt.hist(np.concatenate([d["drops"].numpy() for d in data]), bins=60)
plt.axvline(0, color="k", lw=1)
plt.xlabel("drop in answer logprob when patch is ablated"); plt.ylabel("patches")
plt.title("Ground-truth importance distribution"); plt.tight_layout(); plt.show()

## 6. The table

Every label scored by Spearman against the ground-truth drops. Baselines included so a win has to
be earned: `center` ignores the image entirely, `gaze proximity` is the geometric rival that FRM
must beat for Stage 2b to be worth building.

In [ ]:
def gaze_patch(gz):
    return min(G - 1, int(gz["y_norm"] * G)) * G + min(G - 1, int(gz["x_norm"] * G))

def dist_map(center):
    r0, c0 = divmod(center, G)
    return torch.tensor([-math.hypot(i // G - r0, i % G - c0) for i in range(L_v)])

# leave-one-out baselines for each row-source (never let an example correct itself)
base_q = VS.make_baseline_loo([d["imp_q"] for d in data])
base_a = VS.make_baseline_loo([d["imp_a"] for d in data])

rows = defaultdict(list)
for i, d in enumerate(data):
    gp = gaze_patch(d["gaze"]); d["gaze_patch"] = gp
    pmi_q = VS.pmi_scores(d["imp_q"], base_q[i])
    pmi_a = VS.pmi_scores(d["imp_a"], base_a[i])
    d["pmi_q"], d["pmi_a"] = pmi_q, pmi_a

    cands = {
        "imp_q  raw":            d["imp_q"],
        "imp_q  subtract (old)": VS.subtract_baseline(d["imp_q"], base_q[i]),
        "imp_q  pmi (new)":      pmi_q,
        "imp_a  raw":            d["imp_a"],
        "imp_a  subtract (old)": VS.subtract_baseline(d["imp_a"], base_a[i]),
        "imp_a  pmi (new)":      pmi_a,
        "imp_context (a - q)":   torch.relu(pmi_a - pmi_q),
        "center (no image)":     dist_map(L_v // 2),
        "gaze proximity":        dist_map(gp),
    }
    truth = d["drops"].numpy()
    for k, v in cands.items():
        rows[k].append(spearmanr(v.numpy(), truth)[0])

print(f"Spearman vs ground-truth answer-logprob drops   (n={len(data)})\n")
print(f"{'label':<24}{'mean':>8}{'std':>8}{'>0':>7}")
print("-" * 47)
order = sorted(rows, key=lambda k: -np.nanmean(rows[k]))
for k in order:
    a = np.array(rows[k], dtype=float)
    print(f"{k:<24}{np.nanmean(a):>8.3f}{np.nanstd(a):>8.3f}{(a > 0).sum():>5}/{len(a)}")

## 7. Exp 2a — does answer-importance reach farther than question-grounding?

The FRM premise is **relevance != proximity**: the context the answer needs is often spatially far
from where the eye is. Expected distance from the gaze patch under each importance distribution.

If `imp_answer` does not reach farther than `imp_question`, FRM has nothing to retrieve that the
fovea does not already cover, and Stage 2b should be dropped for the geometric baseline.

In [ ]:
def expected_gaze_distance(scores, gp, cand_mask=None):
    s = scores.clone()
    if cand_mask is not None:
        s[~cand_mask] = -float("inf")
    p = torch.softmax(s, dim=0)
    return float((p * -dist_map(gp)).sum())

eq, ea = [], []
for i, d in enumerate(data):
    gp = d["gaze_patch"]
    eq.append(expected_gaze_distance(d["pmi_q"], gp))
    ea.append(expected_gaze_distance(d["pmi_a"], gp))
eq, ea = np.array(eq), np.array(ea)

print("expected distance from gaze (grid patches; higher = reaches farther)")
print(f"   imp_question : {eq.mean():.2f}")
print(f"   imp_answer   : {ea.mean():.2f}")
print(f"   difference   : {ea.mean() - eq.mean():+.2f}   "
      f"(answer farther in {(ea > eq).sum()}/{len(ea)} examples)")

by = defaultdict(list)
for i, d in enumerate(data):
    by[d["type"]].append((eq[i], ea[i]))
print("\nper question type (question -> answer):")
for t in sorted(by):
    q, a = np.mean([x[0] for x in by[t]]), np.mean([x[1] for x in by[t]])
    flag = "  <- far context" if a - q > 0.5 else ""
    print(f"   {t:<38} {q:4.2f} -> {a:4.2f}{flag}")

plt.figure(figsize=(4.6, 4.4))
plt.scatter(eq, ea, s=30)
lim = [min(eq.min(), ea.min()) - 0.3, max(eq.max(), ea.max()) + 0.3]
plt.plot(lim, lim, "k--", lw=1)
plt.xlabel("imp_question: E[dist from gaze]"); plt.ylabel("imp_answer: E[dist from gaze]")
plt.title("above the line = answer needs far context"); plt.tight_layout(); plt.show()

## 8. See it — ground truth vs the labels

In [ ]:
from PIL import Image as _I

n_show = min(6, len(data))
fig, axes = plt.subplots(n_show, 5, figsize=(15, 2.9 * n_show))
cols = ["image", "GROUND TRUTH (logprob drop)", "imp_q pmi", "imp_a pmi", "imp_context"]
for r in range(n_show):
    d = data[r]
    img = S.load_image(d["img_path"]); W, H = img.size
    panels = [None, d["drops"], d["pmi_q"], d["pmi_a"], torch.relu(d["pmi_a"] - d["pmi_q"])]
    for c in range(5):
        ax = axes[r, c]; ax.imshow(img); ax.axis("off")
        if c > 0:
            h = panels[c].reshape(G, G).numpy()
            h = (h - h.min()) / (np.ptp(h) + 1e-9)          # np.ptp: ndarray.ptp() is gone in numpy 2
            ax.imshow(np.array(_I.fromarray((h * 255).astype("uint8")).resize((W, H))),
                      cmap="jet", alpha=0.5)
        ax.scatter([d["gaze"]["x_norm"] * W], [d["gaze"]["y_norm"] * H],
                   marker="x", s=110, c="lime", linewidths=2.5)
        if r == 0:
            ax.set_title(cols[c], fontsize=9)
    axes[r, 0].set_title(f"[{d['type']}] {d['question'][:60]}", fontsize=7)
plt.tight_layout(); plt.show()

## 9. How to read this

**Section 6 is the verdict.** Decision rules, set before looking:

* **Every label near 0** -> attention-derived importance is not an attribution on this data.
  Do not generate labels at scale. Either switch the teacher to LOO/attention-rollout, or stop.
* **`center` or `gaze proximity` at the top** -> the labels are being beaten by constants and
  geometry. Same conclusion: FRM has nothing to add over the eccentricity baseline.
* **`imp_a` above `imp_q`** -> confirms the spec: answer rows are the training label, question
  rows are the control. Fix Blocker 3 by switching the pipeline to answer rows.
* **`pmi` vs `subtract`** -> whichever wins here settles the argument the synthetic notebook
  could not. Judged against real attribution, so this result stands.

**Section 7 is the precondition.** If `imp_answer` does not reach farther from gaze than
`imp_question`, the data lacks far-context questions, and Exp 1 will look flat no matter how good
FRM is. The per-type breakdown tells you which question types carry the signal — the earlier gaze
eval suggested counting / spatial / next-state are the far-context ones, so check whether that
reproduces here.

**Caveats worth stating in any writeup**

* n=20 is a pilot. Raise `N_PER_TYPE` once the pipeline runs clean; `GROUP=2` makes that ~3x cheaper.
* Ablation is done by masking attention to the patch. A patch can look unimportant simply because a
  neighbour carries redundant information — single-patch LOO understates importance for spatially
  redundant content. Grouped ablation (`GROUP=2`) partly addresses this.
* The scoring target is the gold answer, not the model's own generation. It measures which patches
  support the *correct* answer. Filter to examples with high `base_logp` if you want to restrict to
  cases the model actually answers well.